In [26]:
import os

print(os.listdir("/kaggle/working/"))

['.virtual_documents', 'asl_model.pth']


In [17]:
import torch
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from torchvision import transforms
from PIL import Image

In [9]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
print("before dataset")
train_data = datasets.ImageFolder(
    root="/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train",
    transform=transform,  # change this
)
print("after dataset")
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

print(train_data.classes)  # should show A-Z

before dataset
after dataset
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [13]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(pretrained=True)

# replace last layer for 26 classes
model.fc = nn.Linear(model.fc.in_features, 29)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

cuda


In [14]:
from tqdm import tqdm
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # update progress bar with loss
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} done")

Epoch 1: 100%|██████████| 2719/2719 [16:49<00:00,  2.69it/s, loss=0.0308]  


Epoch 1 done


Epoch 2: 100%|██████████| 2719/2719 [09:38<00:00,  4.70it/s, loss=0.487]   

Epoch 2 done


In [16]:
torch.save(model.state_dict(), "asl_model.pth")

# Load Model

In [69]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 29)  # must match training

state_dict = torch.load("/kaggle/working/asl_model.pth", map_location=device)
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()
print("loaded")

loaded


# Load Dataset

In [71]:
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

test_dir = "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_test/asl_alphabet_test"

image_paths = sorted([
    p for p in glob.glob(os.path.join(test_dir, "*"))
    if os.path.isfile(p)
])

print("num test images:", len(image_paths))
print("first 5:", [os.path.basename(p) for p in image_paths[:5]])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # add Normalize here only if you used it during training
])

class TestImages(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, os.path.basename(path)

test_data = TestImages(image_paths, transform=transform)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

num test images: 28
first 5: ['A_test.jpg', 'B_test.jpg', 'C_test.jpg', 'D_test.jpg', 'E_test.jpg']


In [72]:
all_preds = []
all_files = []

model.eval()

with torch.no_grad():
    for images, names in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_files.extend(names)

print("done")
print(list(zip(all_files[:10], all_preds[:10])))

done
[('A_test.jpg', np.int64(0)), ('B_test.jpg', np.int64(1)), ('C_test.jpg', np.int64(2)), ('D_test.jpg', np.int64(3)), ('E_test.jpg', np.int64(4)), ('F_test.jpg', np.int64(5)), ('G_test.jpg', np.int64(6)), ('H_test.jpg', np.int64(7)), ('I_test.jpg', np.int64(8)), ('J_test.jpg', np.int64(9))]


In [73]:
class_names = val_data.classes  # from your ImageFolder on the train folder
pred_labels = [class_names[i] for i in all_preds]

print(list(zip(all_files[:10], pred_labels[:10])))

[('A_test.jpg', 'A'), ('B_test.jpg', 'B'), ('C_test.jpg', 'C'), ('D_test.jpg', 'D'), ('E_test.jpg', 'E'), ('F_test.jpg', 'F'), ('G_test.jpg', 'G'), ('H_test.jpg', 'H'), ('I_test.jpg', 'I'), ('J_test.jpg', 'J')]


In [75]:
import pandas as pd

submission = pd.DataFrame({
    "file": all_files,
    "label": pred_labels
})

submission.to_csv("/kaggle/working/asl_predictions.csv", index=False)
print("saved to /kaggle/working/asl_predictions.csv")

saved to /kaggle/working/asl_predictions.csv
